In [ ]:
!pip install -q unsloth trl wandb requests pydantic
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import os
if not os.path.exists('dead-internet-detective'):
    !git clone https://github.com/aj7075/dead-internet-detective.git
os.chdir('dead-internet-detective')
!pip install -q -r requirements.txt

In [ ]:
import os
HF_SPACE_URL      = os.getenv("HF_SPACE_URL", "http://localhost:8000")
WANDB_KEY         = os.getenv("WANDB_KEY", "your_wandb_key")
MODEL_NAME        = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
SMOKE_TEST        = True
NUM_TRAIN_STEPS   = 10  if SMOKE_TEST else 150
ROLLOUTS_PER_STEP = 2   if SMOKE_TEST else 4
MAX_EP_STEPS      = 10  if SMOKE_TEST else 30
MAX_EP_SECS       = 90
DIFFICULTIES      = ["easy"] if SMOKE_TEST else ["easy","easy","medium"]
SAVE_PATH         = "./trained_model"
MAX_SEQ_LENGTH    = 1024   # reduced from 2048 for 8GB VRAM


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    use_gradient_checkpointing="unsloth",
)
tokenizer.pad_token = tokenizer.eos_token
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found. Check WSL2 CUDA setup.")
print("Model loaded.")


In [ ]:
import sys, os, json, time, random, requests
sys.path.insert(0, os.path.abspath(".."))
from client.client import DeadInternetClient

client = DeadInternetClient(HF_SPACE_URL)
assert client.health(), f"Server not reachable at {HF_SPACE_URL}"
print(f"Server OK: {HF_SPACE_URL}")

SYSTEM_PROMPT = """You are a disinformation analyst.
At each step respond ONLY with a single JSON object like:
{\"tool\": \"<tool_name>\", \"params\": {<params>}}

Valid tools and their required params:
visit_page:           {\"url\": \"<url>\"}
search:               {\"query\": \"<string>\"}
wayback_check:        {\"url\": \"<url>\"}
whois_lookup:         {\"domain\": \"<domain>\"}
author_lookup:        {\"author_name\": \"<name>\"}
image_provenance:     {\"image_url\": \"<url>\"}
citation_trace:       {\"url\": \"<url>\"}
linguistic_fingerprint: {\"url\": \"<url>\"}
cross_reference:      {\"url_a\": \"<url>\", \"url_b\": \"<url>\"}
update_case_file:     {\"confirmed_facts\": [], \"synthetic_sources\": [],
                       \"credible_sources\": [], \"confidence\": 0.0}
request_expert:       {\"domain\": \"medical|legal|statistical\", \"question\": \"<q>\"}
file_report:          {\"verdict\": \"true|false|unverifiable\",
                       \"confidence\": 0.0,
                       \"evidence_chain\": [\"<url>\"],
                       \"reasoning\": \"<string>\"}

Always end every episode by calling file_report.
Investigate before filing â€” use citation_trace and whois_lookup."""

def generate_action(model, tokenizer, obs_dict):
    prompt = (
        f"<s>[INST] {SYSTEM_PROMPT}\n\n"
        f"OBSERVATION:\n{json.dumps(obs_dict, default=str)[:3000]}\n\n"
        f"Respond with one JSON action: [/INST]"
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.empty_cache()
    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    try:
        start = decoded.find("{")
        end   = decoded.rfind("}") + 1
        return json.loads(decoded[start:end])
    except Exception:
        return {
            "tool": "file_report",
            "params": {
                "verdict": "unverifiable",
                "confidence": 0.3,
                "evidence_chain": [],
                "reasoning": f"parse error: {decoded[:100]}"
            }
        }

def run_episode(model, tokenizer, difficulty="easy", seed=None):
    if seed is None:
        seed = random.randint(0, 99999)
    try:
        reset_result = client.reset(task_id=difficulty, seed=seed)
        session_id   = reset_result["session_id"]
        obs          = reset_result["observation"]
        total_reward = 0.0
        steps        = 0
        t0           = time.time()

        for _ in range(MAX_EP_STEPS):
            if time.time() - t0 > MAX_EP_SECS:
                client.step(session_id, "file_report", {
                    "verdict": "unverifiable", "confidence": 0.3,
                    "evidence_chain": [], "reasoning": "timeout"
                })
                break

            action = generate_action(model, tokenizer, obs)
            result = client.step(session_id, action["tool"], action.get("params", {}))
            total_reward += result.get("reward", 0.0)
            steps        += 1
            obs           = result.get("observation", obs)

            if result.get("done"):
                break

        breakdown = {"total_reward": total_reward, "steps_used": steps,
                     "difficulty": difficulty, "seed": seed,
                     "verdict_accuracy": 0.0, "evidence_chain_quality": 0.0,
                     "synthetic_detection_rate": 0.0,
                     "internal_consistency": 0.0, "step_efficiency": 0.0}
        try:
            state = requests.get(
                f"{HF_SPACE_URL}/state/{session_id}", timeout=5
            ).json()
        except Exception:
            pass
        return breakdown

    except Exception as e:
        print(f"Episode error: {e}")
        return {"total_reward": 0.0, "steps_used": 0, "difficulty": difficulty,
                "seed": seed, "verdict_accuracy": 0.0,
                "evidence_chain_quality": 0.0, "synthetic_detection_rate": 0.0,
                "internal_consistency": 0.0, "step_efficiency": 0.0}


In [ ]:
import statistics

print("=" * 55)
print("SMOKE TEST -- validating pipeline before training")
print("=" * 55)

smoke_results = []
for i in range(10):
    r = run_episode(model, tokenizer, difficulty="easy", seed=i)
    smoke_results.append(r)
    print(f"  ep={i:02d} reward={r['total_reward']:.3f} steps={r['steps_used']}")

rewards = [r["total_reward"] for r in smoke_results]
mean_r  = statistics.mean(rewards)
std_r   = statistics.stdev(rewards) if len(rewards) > 1 else 0.0

print(f"\nMean reward : {mean_r:.3f} +/- {std_r:.3f}")
print(f"Min / Max   : {min(rewards):.3f} / {max(rewards):.3f}")
print(f"Zero rewards: {sum(1 for r in rewards if r == 0.0)}/10")

if mean_r < 0.01:
    raise RuntimeError(
        "SMOKE TEST FAILED: mean reward is near zero. "
        "Check server is running and file_report is being called."
    )
print("\nSmoke test PASSED. Safe to proceed to training.")

In [ ]:
import wandb
from trl import GRPOTrainer, GRPOConfig

wandb.login(key=WANDB_KEY)
wandb.init(project="dead-internet-detective", name="phase1-easy")

# Reward function wrapper for GRPOTrainer
# GRPOTrainer calls this with a list of prompts and generated responses
# We run each as a full environment episode and return the scalar reward

def grpo_reward_fn(prompts, responses, **kwargs):
    rewards = []
    for i, (prompt, response) in enumerate(zip(prompts, responses)):
        difficulty = DIFFICULTIES[i % len(DIFFICULTIES)]
        seed = random.randint(0, 99999)
        result = run_episode(model, tokenizer, difficulty=difficulty, seed=seed)
        r = result["total_reward"]
        rewards.append(float(r))
        wandb.log({
            "train/total_reward":              r,
            "train/verdict_accuracy":          result["verdict_accuracy"],
            "train/evidence_chain_quality":    result["evidence_chain_quality"],
            "train/synthetic_detection_rate":  result["synthetic_detection_rate"],
            "train/internal_consistency":      result["internal_consistency"],
            "train/step_efficiency":           result["step_efficiency"],
            "train/steps_used":                result["steps_used"],
        })
    return rewards

# Dataset: simple prompts -- the actual episode is generated server-side
from datasets import Dataset
prompts = [
    {"prompt": f"Investigate claim #{i}. Difficulty: {DIFFICULTIES[i % len(DIFFICULTIES)]}."}
    for i in range(NUM_TRAIN_STEPS * ROLLOUTS_PER_STEP)
]
dataset = Dataset.from_list(prompts)

training_args = GRPOConfig(
    output_dir=SAVE_PATH,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=ROLLOUTS_PER_STEP,
    learning_rate=2e-5,
    logging_steps=1,
    save_steps=50,
    report_to="wandb",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    max_grad_norm=0.3,
    dataloader_num_workers=0,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=grpo_reward_fn,
    tokenizer=tokenizer,
)

torch.cuda.empty_cache()
import gc; gc.collect()
print(f"Starting GRPO training: {NUM_TRAIN_STEPS} steps, "
      f"{ROLLOUTS_PER_STEP} rollouts/step")
trainer.train()

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")
wandb.finish()


In [ ]:
import json, statistics, os

print("=" * 60)
print("POST-TRAINING EVALUATION")
print("=" * 60)

# Load trained model for eval
from unsloth import FastLanguageModel
eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name=SAVE_PATH,
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(eval_model)

# Baseline: dumb agent (no model, hardcoded)
def run_dumb_episode(difficulty="easy", seed=42):
    try:
        r = client.reset(task_id=difficulty, seed=seed)
        sid = r["session_id"]
        obs = r["observation"]
        total = 0.0
        for url in obs.get("dossier_urls", []):
            res = client.step(sid, "visit_page", {"url": url})
            total += res.get("reward", 0.0)
            if res.get("done"): break
        res = client.step(sid, "file_report", {
            "verdict": "true", "confidence": 0.5,
            "evidence_chain": obs.get("dossier_urls", []),
            "reasoning": "Visited dossier. No contradictions found."
        })
        total += res.get("reward", 0.0)
        return total
    except Exception as e:
        print(f"Dumb agent error: {e}")
        return 0.0

# Run 20 eval episodes each
EVAL_SEEDS   = list(range(100, 120))
baseline_rewards = [run_dumb_episode("easy", s) for s in EVAL_SEEDS]
trained_rewards  = [
    run_episode(eval_model, eval_tokenizer, "easy", s)["total_reward"]
    for s in EVAL_SEEDS
]

b_mean = statistics.mean(baseline_rewards)
t_mean = statistics.mean(trained_rewards)
improvement = ((t_mean - b_mean) / max(abs(b_mean), 1e-6)) * 100

print(f"\n{'Metric':<30} {'Baseline':>10} {'Trained':>10}")
print("-" * 52)
print(f"{'Mean reward':<30} {b_mean:>10.3f} {t_mean:>10.3f}")
print(f"{'Std reward':<30} {statistics.stdev(baseline_rewards):>10.3f} "
      f"{statistics.stdev(trained_rewards):>10.3f}")
print(f"{'Min reward':<30} {min(baseline_rewards):>10.3f} "
      f"{min(trained_rewards):>10.3f}")
print(f"{'Max reward':<30} {max(baseline_rewards):>10.3f} "
      f"{max(trained_rewards):>10.3f}")
print(f"\nImprovement: {improvement:+.1f}%")

# Save results to file
results = {
    "baseline_mean": b_mean,
    "trained_mean":  t_mean,
    "improvement_pct": improvement,
    "baseline_rewards": baseline_rewards,
    "trained_rewards":  trained_rewards,
}
os.makedirs("plots", exist_ok=True)
with open("plots/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nResults saved to plots/eval_results.json")

# Auto-optimization check
print("\n" + "=" * 60)
print("AUTO-OPTIMIZATION ANALYSIS")
print("=" * 60)

suggestions = []

if t_mean <= b_mean:
    suggestions.append(
        "CRITICAL: Trained model is no better than baseline.\n"
        "  Cause: likely too few training steps or reward not propagating.\n"
        "  Fix: set SMOKE_TEST=False, NUM_TRAIN_STEPS=200, "
        "ROLLOUTS_PER_STEP=8 and retrain."
    )

if t_mean < 0.4:
    suggestions.append(
        "Reward is below 0.4 -- model has not learned to investigate.\n"
        "  Fix: increase NUM_TRAIN_STEPS to 300 and add medium difficulty "
        "episodes to DIFFICULTIES list."
    )

if statistics.stdev(trained_rewards) < 0.05:
    suggestions.append(
        "Very low reward variance -- model may be collapsing to one behavior.\n"
        "  Fix: increase temperature to 0.9 in generate_action(), "
        "add more seed diversity."
    )

if improvement > 20:
    suggestions.append(
        "Good improvement detected (>20%). "
        "Ready for Phase 2: add medium difficulty episodes.\n"
        "  Next step: set DIFFICULTIES = ['easy','easy','medium'] and retrain."
    )

if not suggestions:
    suggestions.append("Results look reasonable. No immediate fixes required.")

print("\nRECOMMENDATIONS FOR NEXT RUN:")
for i, s in enumerate(suggestions, 1):
    print(f"\n{i}. {s}")

# Write optimized config suggestions to file
with open("plots/next_run_config.txt", "w") as f:
    f.write("SUGGESTED CONFIG FOR NEXT TRAINING RUN\n")
    f.write("=" * 40 + "\n\n")
    for s in suggestions:
        f.write(s + "\n\n")
print("\nNext-run suggestions saved to plots/next_run_config.txt")

In [ ]:
import json, os
import matplotlib.pyplot as plt

with open("plots/eval_results.json") as f:
    results = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(results["baseline_rewards"], label="Baseline", alpha=0.7, color="red")
axes[0].plot(results["trained_rewards"],  label="Trained",  alpha=0.7, color="green")
axes[0].axhline(results["baseline_mean"], linestyle="--", color="red",   alpha=0.5)
axes[0].axhline(results["trained_mean"],  linestyle="--", color="green", alpha=0.5)
axes[0].set_title("Reward per Episode: Baseline vs Trained")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(["Baseline", "Trained"],
            [results["baseline_mean"], results["trained_mean"]],
            color=["red", "green"], alpha=0.7)
axes[1].set_title(f"Mean Reward Comparison\n"
                  f"Improvement: {results['improvement_pct']:+.1f}%")
axes[1].set_ylabel("Mean Reward")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("plots/reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plots/reward_curve.png")